# Étape 3 — Manipulations : interroger le graphe avec SPARQL

Ce notebook accompagne le `README.md`. On construit nos requêtes **progressivement** : d'abord un seul triplet, puis on ajoute des conditions, et enfin un `FILTER`.

> Prérequis : `pip install -r requirements.txt`. Exécute les cellules dans l'ordre.

## 1. Charger le graphe et un petit utilitaire d'affichage

On charge `personnes.ttl` et on définit une fonction `afficher(requete)` qui exécute une requête SPARQL et affiche joliment les résultats. On la réutilisera dans toutes les cellules suivantes.

In [ ]:
from rdflib import Graph

graphe = Graph()
graphe.parse("personnes.ttl", format="turtle")
print(f"{len(graphe)} triplets charges.\n")


def afficher(requete: str) -> None:
    """Execute une requete SPARQL et affiche chaque ligne de resultat."""
    resultats = graphe.query(requete)
    if not resultats:
        print("  (aucun resultat)")
        return
    for ligne in resultats:
        valeurs = {str(var): str(val) for var, val in ligne.asdict().items()}
        print(" ", valeurs)

22 triplets charges.



## 2. La requête la plus simple : un seul triplet

On cherche tout ce qui est de type `foaf:Person`. La variable `?personne` prend tour à tour chaque valeur qui satisfait le motif.

In [ ]:
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?personne
WHERE {
    ?personne a foaf:Person .
}
""")

 {'personne': 'http://exemple.org/alice'}
  {'personne': 'http://exemple.org/bob'}
  {'personne': 'http://exemple.org/carol'}


## 3. Ajouter un 2e triplet (la "jointure")

On veut aussi le nom. On ajoute un triplet qui réutilise la **même variable** `?personne`. Les deux triplets doivent être vrais en même temps : c'est l'équivalent d'un « ET ».

In [ ]:
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?personne ?nom
WHERE {
    ?personne a foaf:Person .
    ?personne foaf:name ?nom .
}
""")

  {'personne': 'http://exemple.org/alice', 'nom': 'Alice Martin'}
  {'personne': 'http://exemple.org/bob', 'nom': 'Bob Dupont'}
  {'personne': 'http://exemple.org/carol', 'nom': 'Carol Nguyen'}


## 4. Filtrer : les personnes de plus de 30 ans

On récupère l'âge dans `?age`, puis on garde seulement les lignes qui passent la condition avec `FILTER`. Comme l'âge est un entier typé (étape 2), la comparaison `> 30` fonctionne.

In [ ]:
afficher("""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?age
WHERE {
    ?personne foaf:name ?nom .
    ?personne foaf:age ?age .
    FILTER (?age > 30)
}
""")

  {'nom': 'Alice Martin', 'age': '32'}
  {'nom': 'Carol Nguyen', 'age': '41'}


## 5. Expérimente toi-même

Modifie la valeur du `FILTER` ci-dessous (par exemple `< 30`, `>= 27`, `= 41`...) et relance la cellule pour voir comment les résultats changent. C'est le meilleur moyen de prendre SPARQL en main.

In [ ]:
seuil = 30  # <-- change cette valeur et relance la cellule

afficher(f"""
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?nom ?age
WHERE {{
    ?personne foaf:name ?nom .
    ?personne foaf:age ?age .
    FILTER (?age < {seuil})
}}
""")
# Note : en f-string, on double les accolades {{ }} pour ecrire de vraies accolades SPARQL.

  {'nom': 'Bob Dupont', 'age': '27'}


## 6. À toi de jouer : les employeurs

**Exercice :** écris une requête qui retourne **le nom de chaque personne et le nom de son employeur**.

Indices : la relation est `schema:worksFor`, et le nom d'une organisation est `schema:name`. N'oublie pas de déclarer le préfixe `schema:`.

In [ ]:
# TODO : complete cette requete
afficher("""
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX schema: <https://schema.org/>

SELECT ?nomPersonne ?nomEmployeur
WHERE {
    ?personne foaf:name ?nomPersonne .
    # ... a completer ...
}
""")

  {'nomPersonne': 'Alice Martin'}
  {'nomPersonne': 'Bob Dupont'}
  {'nomPersonne': 'Carol Nguyen'}


<details>
<summary>Afficher la solution</summary>

```python
afficher("""
PREFIX foaf:   <http://xmlns.com/foaf/0.1/>
PREFIX schema: <https://schema.org/>

SELECT ?nomPersonne ?nomEmployeur
WHERE {
    ?personne foaf:name ?nomPersonne .
    ?personne schema:worksFor ?employeur .
    ?employeur schema:name ?nomEmployeur .
}
""")
```

Résultat attendu : Alice et Bob → « ACME Corporation », Carol → « Globex ».

</details>

---

Tu sais maintenant interroger un graphe. À l'**étape 4**, on ira plus loin avec `FILTER`, `OPTIONAL`, `BIND` et les fonctions sur chaînes/dates.